# DS6410 Machine Learning II — Lab: Neural Networks
**Course:** DS6410 Machine Learning II  
**Topic:** Neural Networks — Architecture, Training, Regularization, and Comparison with Classical Methods

---

## Lab Overview

This lab has three parts, each directly demonstrating a key concept from the lecture:

| Part | Topic | Key Concept |
|------|-------|-------------|
| **Part 1** | Universal Approximation | A single hidden layer can approximate any continuous function given enough nodes |
| **Part 2** | MNIST Regularization Benchmark | Regularization techniques (Dropout, Max-Norm, Weight Decay) improve generalization |
| **Part 3** | Deep NN vs. Traditional ML | Face recognition: comparing Eigenfaces (PCA + Ridge) vs. SVM vs. Deep MLP |

**Tools:** Python 3, PyTorch, scikit-learn, Matplotlib  
**Estimated Time:** 90–120 minutes

---

> **Instructor Note:** Students should run cells sequentially. Parts 2 and 3 involve training loops that may take 5–10 minutes each on CPU. Encourage students to read the discussion questions while waiting.


## Setup: Install and Import Dependencies

In [ ]:
# Run this cell first to install all required packages
# (Skip if already installed in your environment)
import subprocess, sys
packages = ['torch', 'torchvision', 'scikit-learn', 'matplotlib', 'numpy']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print("All packages ready.")


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, random_split
from torchvision import datasets, transforms
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")


---
## Part 1: The Universal Approximation Theorem

### Background

The **Universal Approximation Theorem** (Cybenko, 1989; Hornik, 1991) states:

> A feedforward neural network with a single hidden layer containing a finite number of neurons can approximate any continuous function on a compact subset of $\mathbb{R}^n$, to any desired degree of accuracy.

This is one of the most important theoretical results in neural network theory. However, it is an **existence** result — it tells us such a network *exists*, but not how to find it, nor how many neurons are needed (which could be exponentially large).

**In this part**, we will empirically verify this theorem by training shallow networks of increasing width to approximate a known, non-trivial function:

$$f(x) = \sin(2\pi x) + 0.5 \sin(4\pi x), \quad x \in [0, 1]$$

We will observe how the quality of approximation improves as we add more hidden nodes.

---
### 1.1 Generate Data


In [ ]:
# --- True function ---
def true_function(x):
    """The target function we want to approximate."""
    return np.sin(2 * np.pi * x) + 0.5 * np.sin(4 * np.pi * x)

# --- Training data (noisy observations) ---
N_TRAIN = 80
x_train = np.random.uniform(0, 1, N_TRAIN).astype(np.float32)
y_train = true_function(x_train) + np.random.normal(0, 0.1, N_TRAIN).astype(np.float32)

# --- Dense grid for plotting the true function and predictions ---
x_plot = np.linspace(0, 1, 300).astype(np.float32)

# Convert to PyTorch tensors
X_t = torch.tensor(x_train).unsqueeze(1)
Y_t = torch.tensor(y_train).unsqueeze(1)
X_plot_t = torch.tensor(x_plot).unsqueeze(1)

# Visualize the data
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(x_train, y_train, s=20, alpha=0.5, color='steelblue', label='Noisy training data')
ax.plot(x_plot, true_function(x_plot), 'k--', linewidth=2, label='True function $f(x)$')
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.set_title('Target Function: $f(x) = \\sin(2\\pi x) + 0.5\\sin(4\\pi x)$')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Training samples: {N_TRAIN}")


### 1.2 Define and Train Shallow Networks of Increasing Width

We define a simple **one-hidden-layer** network:

$$f(x) = W^{(2)} \cdot \text{ReLU}(W^{(1)} x + b^{(1)}) + b^{(2)}$$

We will train networks with **2, 4, 8, 32, and 128** hidden nodes and observe how the approximation quality changes.


In [ ]:
# --- Model definition ---
class ShallowNet(nn.Module):
    """A single-hidden-layer neural network."""
    def __init__(self, n_hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, 1)
        )
    
    def forward(self, x):
        return self.net(x)

# --- Training function ---
def train_shallow_net(n_hidden, epochs=3000, lr=0.01):
    """Train a shallow network and return the trained model."""
    model = ShallowNet(n_hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    for epoch in range(epochs):
        optimizer.zero_grad()
        predictions = model(X_t)
        loss = loss_fn(predictions, Y_t)
        loss.backward()
        optimizer.step()
    
    return model

# --- Train networks of increasing width ---
hidden_sizes = [2, 4, 8, 32, 128]
trained_models = {}

print("Training shallow networks...")
for h in hidden_sizes:
    model = train_shallow_net(h)
    trained_models[h] = model
    with torch.no_grad():
        mse = F.mse_loss(model(X_t), Y_t).item()
    print(f"  {h:4d} hidden nodes — Training MSE: {mse:.4f}")

print("Done.")


### 1.3 Visualize the Approximations

In [ ]:
fig, axes = plt.subplots(1, len(hidden_sizes), figsize=(18, 4), sharey=True)
fig.suptitle("Universal Approximation: Shallow Networks of Increasing Width", 
             fontsize=14, fontweight='bold')

for ax, h in zip(axes, hidden_sizes):
    model = trained_models[h]
    with torch.no_grad():
        y_pred = model(X_plot_t).numpy().flatten()
    
    ax.scatter(x_train, y_train, s=15, alpha=0.4, color='steelblue', label='Training data')
    ax.plot(x_plot, true_function(x_plot), 'k--', linewidth=2, label='True function')
    ax.plot(x_plot, y_pred, 'r-', linewidth=2, label='NN prediction')
    ax.set_title(f'{h} hidden node{"s" if h > 1 else ""}', fontsize=12)
    ax.set_xlabel('x')
    if ax == axes[0]:
        ax.set_ylabel('f(x)')
        ax.legend(fontsize=8)
    ax.set_ylim(-2.2, 2.2)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 1.4 Quantitative Comparison

In [ ]:
# Compute test MSE on the dense grid (treating true function as ground truth)
y_true_plot = true_function(x_plot)

print(f"{'Hidden Nodes':>15} | {'Parameters':>12} | {'Test MSE':>10}")
print("-" * 45)
for h in hidden_sizes:
    model = trained_models[h]
    n_params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        y_pred = model(X_plot_t).numpy().flatten()
    mse = np.mean((y_pred - y_true_plot) ** 2)
    print(f"{h:>15} | {n_params:>12} | {mse:>10.4f}")


### Discussion Questions — Part 1

1. **Observation:** At what number of hidden nodes does the approximation become visually convincing? Does this match your intuition?

2. **Theorem vs. Practice:** The Universal Approximation Theorem guarantees that a solution *exists* with enough nodes. What practical challenges prevent us from simply using a very large network?

3. **Depth vs. Width:** This experiment uses a *shallow* (one hidden layer) network. How do you think a *deep* (multiple hidden layer) network with the same total number of parameters would perform? What does the lecture say about this?

4. **Extension:** Modify the `true_function` to something more complex (e.g., `np.abs(np.sin(5*np.pi*x))`). How many hidden nodes are now required for a good approximation?


---
## Part 2: MNIST Regularization Benchmark

### Background

Because neural networks are **universal approximators** with potentially millions of parameters, they have an immense capacity to memorize training data — a phenomenon known as **overfitting**. Regularization techniques constrain the network's effective capacity and force it to learn generalizable patterns.

In this part, we replicate the benchmark from the lecture (inspired by Srivastava et al., 2014, *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*) on the **MNIST handwritten digit classification** task.

We compare four configurations:

| Configuration | Description |
|---|---|
| **Baseline NN** | Standard network, no regularization |
| **+ Dropout (p=0.5)** | Randomly drop 50% of nodes per mini-batch during training |
| **+ Dropout + Max-Norm** | Dropout combined with a per-node weight magnitude constraint |
| **+ Weight Decay (λ=1e-4)** | L2 penalty on all weight matrices |

---
### 2.1 Load the MNIST Dataset


In [ ]:
# --- Data Loading ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

full_train = datasets.MNIST('/tmp/mnist', train=True,  download=True, transform=transform)
test_set   = datasets.MNIST('/tmp/mnist', train=False, download=True, transform=transform)

# Use 10,000 samples for training and 50,000 for validation (to make overfitting more visible)
train_set, val_set = random_split(full_train, [10000, 50000])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print(f"Training samples  : {len(train_set):,}")
print(f"Validation samples: {len(val_set):,}")
print(f"Test samples      : {len(test_set):,}")
print(f"Classes           : 10 (digits 0–9)")
print(f"Input dimension   : 28×28 = 784 pixels")

# Visualize a few samples
fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle("Sample MNIST Images", fontsize=12)
for i, ax in enumerate(axes.flat):
    img, label = train_set[i]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(str(label), fontsize=10)
    ax.axis('off')
plt.tight_layout(); plt.show()


### 2.2 Define the Network Architecture and Training Utilities

In [ ]:
# --- Network Architecture ---
class MNISTNet(nn.Module):
    """
    A 3-layer fully connected network for MNIST classification.
    The dropout_p parameter controls the dropout probability (0 = no dropout).
    """
    def __init__(self, dropout_p=0.0):
        super().__init__()
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
        self.dropout = nn.Dropout(p=dropout_p)
    
    def forward(self, x):
        x = x.view(-1, 784)           # Flatten 28x28 image to 784-dim vector
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.dropout(F.relu(self.fc2(x)))
        return self.fc3(x)            # Raw logits (no softmax; handled by CrossEntropyLoss)

# --- Max-Norm Constraint ---
def apply_max_norm(model, max_norm=3.0):
    """
    Project each hidden node's incoming weight vector onto the L2 ball of radius max_norm.
    Applied after each gradient update step.
    """
    with torch.no_grad():
        for name, param in model.named_parameters():
            if 'weight' in name:
                norms = param.data.norm(2, dim=1, keepdim=True)
                # Scale down only if norm exceeds max_norm
                scale = (max_norm / norms).clamp(max=1.0)
                param.data *= scale

# --- Training and Evaluation Functions ---
def train_epoch(model, loader, optimizer, weight_decay_l2=0.0, use_max_norm=False):
    """Train for one epoch and return the average loss."""
    model.train()
    total_loss = 0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = F.cross_entropy(logits, yb)
        
        # Manual L2 weight decay (added to loss)
        if weight_decay_l2 > 0:
            l2_penalty = sum(p.pow(2).sum() for n, p in model.named_parameters() if 'weight' in n)
            loss = loss + weight_decay_l2 * l2_penalty
        
        loss.backward()
        optimizer.step()
        
        # Apply max-norm constraint after each update
        if use_max_norm:
            apply_max_norm(model)
        
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    """Return the accuracy (%) on a given data loader."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xb, yb in loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return 100.0 * correct / total

print("Model and training utilities defined.")
print(f"Network parameters: {sum(p.numel() for p in MNISTNet().parameters()):,}")


### 2.3 Train All Configurations

> **Note:** This cell trains 4 models for 15 epochs each. On CPU, this takes approximately 5–8 minutes.


In [ ]:
# --- Experiment Configurations ---
configs = [
    {"name": "Baseline NN",             "dropout": 0.0, "wd": 0.0,   "max_norm": False},
    {"name": "+ Dropout (p=0.5)",        "dropout": 0.5, "wd": 0.0,   "max_norm": False},
    {"name": "+ Dropout + Max-Norm",     "dropout": 0.5, "wd": 0.0,   "max_norm": True},
    {"name": "+ Weight Decay (λ=1e-4)",  "dropout": 0.0, "wd": 1e-4,  "max_norm": False},
]

EPOCHS = 15
results = {}

for cfg in configs:
    print(f"Training: {cfg['name']} ...")
    model = MNISTNet(dropout_p=cfg['dropout'])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    train_accs, val_accs = [], []
    for epoch in range(EPOCHS):
        train_epoch(model, train_loader, optimizer,
                    weight_decay_l2=cfg['wd'], use_max_norm=cfg['max_norm'])
        train_accs.append(evaluate(model, train_loader))
        val_accs.append(evaluate(model, val_loader))
    
    test_acc = evaluate(model, test_loader)
    results[cfg['name']] = {"train": train_accs, "val": val_accs, "test": test_acc}
    
    gap = train_accs[-1] - val_accs[-1]
    print(f"  Train: {train_accs[-1]:.2f}% | Val: {val_accs[-1]:.2f}% | "
          f"Test: {test_acc:.2f}% | Train-Val Gap: {gap:.2f}%")

print("\nAll configurations trained.")


### 2.4 Visualize Results

In [ ]:
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MNIST Regularization Benchmark", fontsize=14, fontweight='bold')

# Left: Learning curves
for i, (name, res) in enumerate(results.items()):
    epochs_range = range(1, EPOCHS + 1)
    axes[0].plot(epochs_range, res['train'], '--', color=colors[i], alpha=0.6, linewidth=1.5)
    axes[0].plot(epochs_range, res['val'],   '-',  color=colors[i], label=name, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Training (dashed) vs. Validation (solid) Accuracy')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Right: Final test accuracy bar chart
names = list(results.keys())
test_accs = [results[n]['test'] for n in names]
bars = axes[1].bar(names, test_accs, color=colors, edgecolor='black', width=0.55)
for bar, acc in zip(bars, test_accs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 f'{acc:.2f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylim(min(test_accs) - 2, 100)
axes[1].set_ylabel('Test Accuracy (%)')
axes[1].set_title('Final Test Accuracy by Configuration')
axes[1].tick_params(axis='x', rotation=15)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()


### 2.5 Summary Table

In [ ]:
print(f"{'Configuration':<30} | {'Train Acc':>10} | {'Val Acc':>8} | {'Test Acc':>9} | {'Gap':>8}")
print("-" * 75)
for name, res in results.items():
    gap = res['train'][-1] - res['val'][-1]
    print(f"{name:<30} | {res['train'][-1]:>9.2f}% | {res['val'][-1]:>7.2f}% | "
          f"{res['test']:>8.2f}% | {gap:>7.2f}%")


### Discussion Questions — Part 2

1. **Overfitting Signal:** Look at the training vs. validation accuracy curves. Which configuration shows the largest gap (most overfitting)? Does this match your expectation?

2. **Dropout Effect:** Dropout typically *slows down* training convergence (the training accuracy rises more slowly). Can you observe this in the curves? Why does this happen?

3. **Max-Norm + Dropout:** The lecture notes that Max-Norm is particularly effective *when combined with Dropout*. What is the intuitive reason for this synergy?

4. **Practical Guidance:** Based on these results, which configuration would you recommend for a practitioner? Consider both final accuracy and the train-val gap.

5. **Extension:** Increase the training set size from 10,000 to 50,000 (use the full training set). How does this change the relative benefit of each regularization technique? What does this tell you about when regularization matters most?


---
## Part 3: Deep NN vs. Traditional ML — Face Recognition

### Background

This part is inspired by two sources:
1. **The Eigenfaces approach** (Turk & Pentland, 1991), discussed in the lecture, which applies **PCA** to face images to extract "eigenfaces" — the principal components of the face image space — and then trains a linear classifier on these compressed representations.
2. **He et al. (2020)**, *"Deep neural networks and kernel regression achieve comparable accuracies for functional connectivity prediction"* (NeuroImage), which found that on structured, moderate-sized datasets, classical ML methods (kernel regression) can match deep neural networks.

**The key question:** Does a deep neural network always outperform classical ML methods? Or does the answer depend on the dataset size and structure?

We use the **Olivetti Faces dataset** (AT&T Laboratories Cambridge):
- 400 grayscale face images (64×64 pixels = 4,096 features)
- 40 subjects, 10 images each
- Task: identify the subject (40-class classification)

We compare three methods using **5-fold cross-validation**:

| Method | Description |
|---|---|
| **PCA + Ridge Regression** | Eigenfaces approach: compress to 80 PCs, then linear classifier |
| **PCA + SVM (RBF kernel)** | Eigenfaces + non-linear kernel classifier |
| **Deep MLP** | End-to-end deep network (no manual feature engineering) |

---
### 3.1 Load and Visualize the Data


In [ ]:
# --- Load the Olivetti Faces Dataset ---
print("Loading Olivetti Faces dataset...")
faces = fetch_olivetti_faces(data_home='/tmp/olivetti', shuffle=True, random_state=42)
X, y = faces.data.astype(np.float32), faces.target

print(f"Dataset shape : {X.shape}  ({X.shape[0]} images × {X.shape[1]} pixels)")
print(f"Subjects      : {len(np.unique(y))} (10 images per subject)")
print(f"Image size    : 64×64 pixels")

# Visualize sample faces
fig, axes = plt.subplots(4, 10, figsize=(16, 7))
fig.suptitle("Olivetti Faces Dataset — 4 Sample Subjects (10 images each)", fontsize=12)
for subject in range(4):
    subject_imgs = X[y == subject]
    for j in range(10):
        axes[subject, j].imshow(subject_imgs[j].reshape(64, 64), cmap='bone')
        axes[subject, j].axis('off')
        if j == 0:
            axes[subject, j].set_ylabel(f'Subject {subject}', fontsize=9)
plt.tight_layout(); plt.show()


### 3.2 Visualize the Eigenfaces (PCA Components)

The **eigenfaces** are the principal components of the face image space. They represent the directions of maximum variance in the data. Intuitively, each eigenface captures a different "mode" of variation across faces (lighting, pose, expression, identity).


In [ ]:
# --- Compute and Visualize Eigenfaces ---
scaler_vis = StandardScaler()
X_scaled_vis = scaler_vis.fit_transform(X)

pca_vis = PCA(n_components=16, whiten=True, random_state=42)
pca_vis.fit(X_scaled_vis)
eigenfaces = pca_vis.components_.reshape((16, 64, 64))

# Explained variance
cumvar = np.cumsum(pca_vis.explained_variance_ratio_) * 100

fig, axes = plt.subplots(2, 8, figsize=(16, 5))
fig.suptitle("The 16 Leading Eigenfaces (Principal Components of Face Space)", fontsize=12, fontweight='bold')
for ax, ef, var in zip(axes.flat, eigenfaces, pca_vis.explained_variance_ratio_ * 100):
    ax.imshow(ef, cmap='bone')
    ax.set_title(f'{var:.1f}%', fontsize=8)
    ax.axis('off')
plt.tight_layout(); plt.show()

# Cumulative variance plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 17), cumvar, 'bo-', linewidth=2, markersize=6)
ax.axhline(y=80, color='r', linestyle='--', label='80% variance')
ax.set_xlabel('Number of Principal Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('Cumulative Variance Explained by Eigenfaces')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Top 16 components explain {cumvar[-1]:.1f}% of total variance.")


### 3.3 Method 1: PCA + Ridge Regression (Eigenfaces)

In [ ]:
def run_pca_ridge(X, y, n_components=80, alpha=1.0):
    """
    Eigenfaces approach:
    1. Standardize pixel values
    2. Project onto top n_components PCA directions (eigenfaces)
    3. Train a Ridge Regression classifier on the compressed representations
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        
        # Step 1: Standardize
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
        
        # Step 2: PCA (eigenfaces)
        pca = PCA(n_components=n_components, whiten=True, random_state=42)
        X_tr = pca.fit_transform(X_tr)
        X_te = pca.transform(X_te)
        
        # Step 3: Ridge Regression classifier
        clf = RidgeClassifier(alpha=alpha)
        clf.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_te, clf.predict(X_te)))
    
    return np.array(fold_accs)

print("Running PCA + Ridge Regression (Eigenfaces)...")
ridge_accs = run_pca_ridge(X, y, n_components=80)
print(f"  Fold accuracies : {[f'{a*100:.1f}%' for a in ridge_accs]}")
print(f"  Mean ± Std      : {ridge_accs.mean()*100:.1f}% ± {ridge_accs.std()*100:.1f}%")


### 3.4 Method 2: PCA + SVM (RBF Kernel)

In [ ]:
def run_pca_svm(X, y, n_components=80, C=10):
    """
    PCA + Support Vector Machine with RBF kernel.
    The RBF kernel implicitly maps features to a high-dimensional space,
    enabling non-linear decision boundaries.
    """
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
        
        pca = PCA(n_components=n_components, whiten=True, random_state=42)
        X_tr = pca.fit_transform(X_tr)
        X_te = pca.transform(X_te)
        
        clf = SVC(kernel='rbf', C=C, gamma='scale')
        clf.fit(X_tr, y_tr)
        fold_accs.append(accuracy_score(y_te, clf.predict(X_te)))
    
    return np.array(fold_accs)

print("Running PCA + SVM (RBF kernel)...")
svm_accs = run_pca_svm(X, y, n_components=80)
print(f"  Fold accuracies : {[f'{a*100:.1f}%' for a in svm_accs]}")
print(f"  Mean ± Std      : {svm_accs.mean()*100:.1f}% ± {svm_accs.std()*100:.1f}%")


### 3.5 Method 3: Deep MLP (End-to-End)

> **Note:** This cell trains 5 models (one per fold). On CPU, this takes approximately 3–5 minutes.


In [ ]:
class DeepMLP(nn.Module):
    """
    A deep fully-connected network for face recognition.
    Uses Batch Normalization and Dropout for regularization.
    Operates directly on raw pixel values (no manual feature engineering).
    """
    def __init__(self, in_dim=4096, n_classes=40):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256),    nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(256, 128),    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )
    
    def forward(self, x):
        return self.net(x)

def run_deep_mlp(X, y, epochs=80):
    """Train a Deep MLP with 5-fold cross-validation."""
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_accs = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        print(f"  Fold {fold+1}/5 ...")
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
        X_te = scaler.transform(X_te)
        
        # Convert to tensors
        X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
        y_tr_t = torch.tensor(y_tr, dtype=torch.long)
        X_te_t = torch.tensor(X_te, dtype=torch.float32)
        
        loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)
        
        model = DeepMLP()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
        # Cosine annealing: gradually reduce learning rate
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
        
        for epoch in range(epochs):
            model.train()
            for xb, yb in loader:
                optimizer.zero_grad()
                F.cross_entropy(model(xb), yb).backward()
                optimizer.step()
            scheduler.step()
        
        model.eval()
        with torch.no_grad():
            preds = model(X_te_t).argmax(dim=1).numpy()
        fold_accs.append(accuracy_score(y_te, preds))
    
    return np.array(fold_accs)

print("Running Deep MLP (end-to-end)...")
mlp_accs = run_deep_mlp(X, y, epochs=80)
print(f"  Fold accuracies : {[f'{a*100:.1f}%' for a in mlp_accs]}")
print(f"  Mean ± Std      : {mlp_accs.mean()*100:.1f}% ± {mlp_accs.std()*100:.1f}%")


### 3.6 Compare All Methods

In [ ]:
methods = ['PCA + Ridge\n(Eigenfaces)', 'PCA + SVM\n(RBF Kernel)', 'Deep MLP\n(End-to-End)']
all_accs = [ridge_accs * 100, svm_accs * 100, mlp_accs * 100]
means = [a.mean() for a in all_accs]
stds  = [a.std()  for a in all_accs]
colors_p3 = ['#3498db', '#e67e22', '#e74c3c']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Deep NN vs. Traditional ML — Olivetti Face Recognition (5-Fold CV)", 
             fontsize=13, fontweight='bold')

# Bar chart with error bars
bars = axes[0].bar(methods, means, yerr=stds, capsize=8, 
                   color=colors_p3, edgecolor='black', width=0.5)
for bar, m, s in zip(bars, means, stds):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.3,
                 f'{m:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_ylim(max(0, min(means) - 10), 105)
axes[0].set_ylabel('5-Fold CV Accuracy (%)')
axes[0].set_title('Mean Accuracy ± Std Dev')
axes[0].grid(True, axis='y', alpha=0.3)

# Box plot — shows the distribution of fold accuracies
bp = axes[1].boxplot(all_accs, tick_labels=methods, patch_artist=True, notch=False,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors_p3):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Distribution of Fold Accuracies')
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

# Summary table
print(f"\n{'Method':<30} | {'Mean Acc':>10} | {'Std Dev':>8} | {'Min Fold':>10} | {'Max Fold':>10}")
print("-" * 75)
for name, accs in zip(['PCA + Ridge (Eigenfaces)', 'PCA + SVM (RBF)', 'Deep MLP'], all_accs):
    print(f"{name:<30} | {accs.mean():>9.1f}% | {accs.std():>7.1f}% | "
          f"{accs.min():>9.1f}% | {accs.max():>9.1f}%")


### Discussion Questions — Part 3

1. **Surprising Result?** The deep MLP operates on raw pixels (4,096 features) with only 400 training samples. Does its performance relative to the classical methods surprise you? What does this suggest about the relationship between dataset size and the advantage of deep learning?

2. **Connection to He et al. (2020):** The paper you referenced found that kernel regression and deep neural networks achieve *comparable* accuracy on brain connectivity data. Does your result here support or contradict this finding? What conditions might favor one approach over the other?

3. **The Eigenfaces Approach:** PCA reduces 4,096 pixel features to 80 principal components. What information is *lost* in this compression? In what scenarios might this lost information be critical?

4. **Data Efficiency:** Deep learning is often said to require large amounts of data. With only 320 training samples (80% of 400), the deep MLP still performs competitively. What architectural choices (BatchNorm, Dropout, weight decay) help it generalize despite limited data?

5. **Practical Recommendation:** Given the results, when would you recommend using a classical ML pipeline (PCA + SVM) over a deep neural network? Consider factors like dataset size, computational budget, interpretability, and performance.


---
## Lab Summary

This lab demonstrated three foundational results in neural network theory and practice:

### Part 1 — Universal Approximation
A single-hidden-layer network with ReLU activations can approximate the function $f(x) = \sin(2\pi x) + 0.5\sin(4\pi x)$ to increasing precision as the number of hidden nodes grows. With only 2 nodes, the network is essentially linear. With 128 nodes, the approximation is nearly perfect. This empirically validates the Universal Approximation Theorem.

### Part 2 — Regularization on MNIST
All regularization techniques reduce the train-validation accuracy gap compared to the baseline, confirming that they constrain the network's effective capacity. The results illustrate the key trade-off: regularization slightly reduces training accuracy but improves generalization. The optimal choice depends on the specific dataset and network architecture.

### Part 3 — Deep NN vs. Traditional ML
On the Olivetti Faces dataset (400 samples, 40 classes), the three methods achieve comparable accuracy. This mirrors the finding of He et al. (2020) that classical methods can match deep neural networks on moderate-sized, structured datasets. Deep learning's advantage becomes most pronounced with large datasets and raw, unstructured inputs (images, text, audio) where manual feature engineering is impractical.

---
### Key Takeaways

| Concept | Takeaway |
|---|---|
| **Universal Approximation** | Width (more nodes) increases approximation power, but depth is more efficient |
| **Overfitting** | Large networks memorize training data; regularization is essential |
| **Dropout** | Effective regularizer; can be viewed as training an exponential ensemble |
| **Deep vs. Classical ML** | Deep learning excels with large datasets and raw inputs; classical ML is competitive on small, structured data |

---
*DS6410 Machine Learning II — Neural Networks Lab*
